In [2]:
import torch 
import torch.nn as nn

In [3]:
class ExampleDeepNeuralNetowrks(nn.Module):
    def __init__(self, layer_size, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_size[0], layer_size[1]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_size[1], layer_size[2]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_size[2], layer_size[3]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_size[3], layer_size[4]), nn.GELU()),
            nn.Sequential(nn.Linear(layer_size[4], layer_size[5]), nn.GELU()),
        ])

    def forward(self,x):
        for layers in self.layers:
            layer_output = layers(x)

            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output

        return x

In [4]:
layer_size = [3, 3 , 3, 3, 3, 1]
sample_input = torch.tensor([
    [1.0, 0.0, -1.0]
])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetowrks(
    layer_size, use_shortcut = False,
)

In [5]:
def print_gradients(model,x):
    output = model(x)
    target = torch.tensor([[0.0]])

    loss = nn.MSELoss()
    loss = loss(output, target)

    loss.backward()

    for name , param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

    

In [6]:
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020174124801997095
layers.1.0.weight has gradient mean of 0.00012011774379061535
layers.2.0.weight has gradient mean of 0.0007152438047342002
layers.3.0.weight has gradient mean of 0.0013988513965159655
layers.4.0.weight has gradient mean of 0.005049603525549173


In [7]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetowrks(
    layer_size, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22186799347400665
layers.1.0.weight has gradient mean of 0.20709271728992462
layers.2.0.weight has gradient mean of 0.3292388319969177
layers.3.0.weight has gradient mean of 0.2667771577835083
layers.4.0.weight has gradient mean of 1.3268064260482788
